# 01 - Define Allen Lab Context

**ChatGPT Track**  
**allen-lab-report-tool**

This notebook loads `src/chatgpt/lab_context.py`, validates the Allen Lab context profile, and exports JSON + Markdown artifacts.

Expected repo layout:

```text
allen-lab-report-tool/
├── src/
│   └── chatgpt/
│       ├── __init__.py
│       ├── lab_context.py
│       ├── schemas.py
│       └── utils.py
└── notebooks/
    └── chatgpt/
        └── 01_define_allen_lab_context.ipynb
```


In [ ]:
# ================================================
# SETUP: Colab + local
# ================================================
from pathlib import Path
import json
import sys
import subprocess

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

# Case 1: running from repo root
if (cwd / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd

# Case 2: running from notebooks/chatgpt inside repo
elif cwd.name == "chatgpt" and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]

# Case 3: Colab default /content with cloned repo
elif (cwd / REPO_NAME / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd / REPO_NAME

# Case 4: Colab default /content with no repo cloned yet
else:
    print("Repo not found in current runtime. Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
chatgpt_path = src_path / "chatgpt"
lab_context_path = chatgpt_path / "lab_context.py"

print("cwd:", cwd)
print("repo_root:", repo_root)
print("src_path:", src_path)
print("chatgpt_path:", chatgpt_path)
print("lab_context exists:", lab_context_path.exists())

if not lab_context_path.exists():
    raise FileNotFoundError(f"Missing expected file: {lab_context_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chatgpt.lab_context import ALLEN_LAB_CONTEXT

print("ChatGPT lab context loaded.")
print("Institution:", ALLEN_LAB_CONTEXT["institution"])


## 1. Inspect Context

In [ ]:
ALLEN_LAB_CONTEXT


## 2. Validate Context Keys

In [ ]:
required_keys = [
    "institution",
    "likely_focus_areas",
    "likely_equipment_or_platforms",
    "report_priorities",
]

missing = [key for key in required_keys if key not in ALLEN_LAB_CONTEXT]

if missing:
    raise ValueError(f"Missing required context keys: {missing}")

for key in required_keys:
    print(f"✓ {key}: {type(ALLEN_LAB_CONTEXT[key]).__name__}")


## 3. Display Context Tables

In [ ]:
import pandas as pd

focus_df = pd.DataFrame({"focus_area": ALLEN_LAB_CONTEXT["likely_focus_areas"]})
equipment_df = pd.DataFrame({"equipment_or_platform": ALLEN_LAB_CONTEXT["likely_equipment_or_platforms"]})
priorities_df = pd.DataFrame({"report_priority": ALLEN_LAB_CONTEXT["report_priorities"]})

display(focus_df)
display(equipment_df)
display(priorities_df)


## 4. Build Context Record

In [ ]:
context_record = {
    **ALLEN_LAB_CONTEXT,
    "generator_track": "chatgpt",
    "source_file": "src/chatgpt/lab_context.py",
    "notebook": "notebooks/chatgpt/01_define_allen_lab_context.ipynb",
}

context_record


## 5. Export JSON

In [ ]:
results_dir = repo_root / "results" / "chatgpt"
reports_dir = repo_root / "reports" / "chatgpt"

results_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

json_path = results_dir / "allen_lab_context.json"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(context_record, f, indent=2, ensure_ascii=False)

print("Wrote:", json_path)


## 6. Export Markdown

In [ ]:
def bullets(items):
    return "\n".join(f"- {item}" for item in items)

md_text = f"""# Allen Lab Context Profile

**Generator track:** ChatGPT  
**Source file:** `src/chatgpt/lab_context.py`  
**Notebook:** `notebooks/chatgpt/01_define_allen_lab_context.ipynb`

## Institution

{context_record['institution']}

## Focus areas

{bullets(context_record['likely_focus_areas'])}

## Equipment or platforms

{bullets(context_record['likely_equipment_or_platforms'])}

## Report priorities

{bullets(context_record['report_priorities'])}
"""

md_path = reports_dir / "allen_lab_context.md"
md_path.write_text(md_text, encoding="utf-8")

print("Wrote:", md_path)


## 7. Confirm Exports

In [ ]:
print(json_path.read_text(encoding="utf-8"))


In [ ]:
print(md_path.read_text(encoding="utf-8"))


## 8. Summary

Notebook 01 loads the ChatGPT context file from `src/chatgpt/`, validates expected fields, and exports reusable artifacts.

**Next:** Notebook 02 can load a paper, page, dataset note, or source text and attach source metadata to this context.
